# Notebook 1 — Suzlon Energy: Bronze → Silver

**Purpose:** Ingest the generated CSV files from the landing volume into logical Bronze tables, then apply explicit schemas, data types, cleaning and quality checks before writing Silver Delta tables.

**Design:** Bronze preserves the data as received by reading all columns as strings. Silver converts types and enforces business-valid ranges. The logical zone is represented through table naming (`bronze_*`, `silver_*`) in the default schema so the notebook also works in restricted/free workspaces.

In [0]:
# Configuration
CATALOG = "workspace"
SCHEMA = "default"
LANDING = "/Volumes/workspace/default/suzlon_landing"

def t(name):
    return f"{CATALOG}.{SCHEMA}.{name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("Landing:", LANDING)


Landing: /Volumes/workspace/default/suzlon_landing


## 1. Bronze ingestion

The raw CSVs are loaded with every field as a string. This is intentional: Bronze is the preservation layer and should not silently change types or business values.

In [0]:
files = [
    "dim_model","dim_customer","dim_site","dim_date","dim_turbine","dim_technician",
    "fact_order","fact_contract","fact_generation","fact_service"
]
for name in files:
    df = (spark.read.format("csv")
          .option("header", True)
          .option("inferSchema", False)
          .load(f"{LANDING}/{name}.csv"))
    df.write.format("delta").mode("overwrite").saveAsTable(t(f"bronze_{name}"))
    print(name, df.count())


dim_model 15
dim_customer 5000
dim_site 300
dim_date 365
dim_turbine 1000
dim_technician 250
fact_order 20000
fact_contract 6000
fact_generation 365000
fact_service 30000


## 2. Silver schemas and transformations

Silver is the typed and validated layer. Monetary values become `DECIMAL`, keys become integers, dates become `DATE`, and percentages/durations become numeric types. Invalid rows are removed only where they violate basic business rules.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

schemas = {
"dim_model": [("model_id","int"),("model_name","string"),("rated_capacity_kw","int")],
"dim_customer": [("customer_id","int"),("customer_name","string"),("segment","string"),("state","string")],
"dim_site": [("site_id","int"),("site_name","string"),("state","string"),("region","string"),("wind_zone_class","string")],
"dim_date": [("date_id","int"),("full_date","date"),("month","int"),("quarter","int"),("year","int"),("fiscal_year","string"),("is_holiday","int")],
"dim_turbine": [("turbine_id","int"),("serial_no","string"),("site_id","int"),("model_id","int"),("commissioning_date","date")],
"dim_technician": [("technician_id","int"),("tech_name","string"),("region","string"),("skill_level","string")],
"fact_order": [("order_id","int"),("customer_id","int"),("model_id","int"),("date_id","int"),("order_value_inr","decimal(18,2)"),("mw_capacity","decimal(10,2)")],
"fact_contract": [("contract_id","int"),("customer_id","int"),("date_id","int"),("contract_value_inr","decimal(18,2)"),("renewed_flag","int")],
"fact_generation": [("gen_id","int"),("turbine_id","int"),("site_id","int"),("date_id","int"),("energy_kwh","decimal(18,2)"),("availability_pct","decimal(6,2)"),("avg_wind_speed_mps","decimal(6,2)"),("uptime_hours","decimal(8,2)"),("downtime_hours","decimal(8,2)")],
"fact_service": [("ticket_id","int"),("turbine_id","int"),("technician_id","int"),("date_id","int"),("downtime_hours","decimal(10,2)"),("ticket_type","string"),("fault_severity_score","int"),("parts_cost_inr","decimal(18,2)")]
}

for name, cols in schemas.items():
    df = spark.table(t(f"bronze_{name}"))
    for col, typ in cols:
        df = df.withColumn(col, F.col(col).cast(typ))
    # Basic trimming for strings
    for col, typ in cols:
        if typ == "string":
            df = df.withColumn(col, F.trim(F.col(col)))
    # Table-specific quality filters
    if name == "dim_model":
        df = df.filter("model_id IS NOT NULL AND rated_capacity_kw > 0")
    elif name == "dim_customer":
        df = df.filter("customer_id IS NOT NULL")
    elif name == "dim_site":
        df = df.filter("site_id IS NOT NULL")
    elif name == "dim_date":
        df = df.filter("date_id IS NOT NULL AND full_date IS NOT NULL")
    elif name == "dim_turbine":
        df = df.filter("turbine_id IS NOT NULL AND site_id IS NOT NULL AND model_id IS NOT NULL")
    elif name == "dim_technician":
        df = df.filter("technician_id IS NOT NULL")
    elif name == "fact_order":
        df = df.filter("order_id IS NOT NULL AND order_value_inr >= 0 AND mw_capacity >= 0")
    elif name == "fact_contract":
        df = df.filter("contract_id IS NOT NULL AND contract_value_inr >= 0 AND renewed_flag IN (0,1)")
    elif name == "fact_generation":
        df = df.filter("gen_id IS NOT NULL AND energy_kwh >= 0 AND availability_pct BETWEEN 0 AND 100")
    elif name == "fact_service":
        df = df.filter("ticket_id IS NOT NULL AND downtime_hours >= 0 AND parts_cost_inr >= 0")
    df = df.dropDuplicates()
    df.write.format("delta").mode("overwrite").saveAsTable(t(f"silver_{name}"))
    print("silver_", name, df.count())


silver_ dim_model 15
silver_ dim_customer 5000
silver_ dim_site 300
silver_ dim_date 365
silver_ dim_turbine 1000
silver_ dim_technician 250
silver_ fact_order 20000
silver_ fact_contract 6000
silver_ fact_generation 365000
silver_ fact_service 30000


## 3. Silver data-quality checks

These checks confirm primary-key uniqueness and the main business ranges before Gold processing.

In [0]:
checks = {
    "silver_dim_model": "model_id IS NULL",
    "silver_dim_customer": "customer_id IS NULL",
    "silver_dim_site": "site_id IS NULL",
    "silver_dim_date": "date_id IS NULL OR full_date IS NULL",
    "silver_dim_turbine": "turbine_id IS NULL OR site_id IS NULL OR model_id IS NULL",
    "silver_dim_technician": "technician_id IS NULL",
    "silver_fact_order": "order_id IS NULL OR order_value_inr < 0",
    "silver_fact_contract": "contract_id IS NULL OR renewed_flag NOT IN (0,1)",
    "silver_fact_generation": "gen_id IS NULL OR availability_pct < 0 OR availability_pct > 100",
    "silver_fact_service": "ticket_id IS NULL OR downtime_hours < 0"
}
for table, condition in checks.items():
    bad = spark.table(t(table)).filter(condition).count()
    print(table, "bad_rows =", bad)


silver_dim_model bad_rows = 0
silver_dim_customer bad_rows = 0
silver_dim_site bad_rows = 0
silver_dim_date bad_rows = 0
silver_dim_turbine bad_rows = 0
silver_dim_technician bad_rows = 0
silver_fact_order bad_rows = 0
silver_fact_contract bad_rows = 0
silver_fact_generation bad_rows = 0
silver_fact_service bad_rows = 0


## 4. Partitioning decision

The largest fact is daily generation. For the Silver Delta table, date-based filtering is the dominant analytical access pattern. In this assignment we keep the physical Silver table simple and use the date dimension for filtering. Gold monthly aggregates reduce the data volume used by the dashboard. This avoids creating many tiny partitions on a small synthetic dataset.